# Setup

All the Imports needed for this experiment.

In [ ]:
from matplotlib import pyplot as plt
from collections import defaultdict
from tqdm import tqdm
import gymnasium as gym
import numpy as np
import pandas as pd
import pickle
import ipywidgets as widgets
from datetime import datetime

import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "env" / "blackjack_env.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from env.blackjack_env import BlackjackEnv
from agents.q_learning_agent import QLearningBlackjackAgent


Define default font size, plot colors etc.

In [ ]:
def setup_plot_style():
    """Definiert das globale Design für alle Matplotlib-Plots."""
    # Schriftgrößen
    plt.rcParams['font.size'] = 11
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10
    
    # Linien & Grid
    plt.rcParams['lines.linewidth'] = 2.0
    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.linestyle'] = '--'
    plt.rcParams['grid.alpha'] = 0.5
    
    # Layout & Render-Qualität
    plt.rcParams['figure.autolayout'] = True
    plt.rcParams['figure.dpi'] = 120
    
    # Optionale Farbpalette (z.B. "Tab10" oder "Set2" für modernen Look)
    plt.style.use('seaborn-v0_8-whitegrid') # Falls ein Grundtheme gewünscht ist

setup_plot_style()
AGENT_STYLES = {    
    "baseline": {"color": "#6b7c93", "label": "Baseline"},
    "counting":  {"color": "#2a9d8f", "label": "Counting"},
}


Create image folder for high-res images and image save function.

In [ ]:
IMAGES_PATH = PROJECT_ROOT / "images"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)
    print(f"Figure saved: {path}")

# Environment & Agents

In [ ]:
def baseline_state_key(obs) -> tuple[int, int, int]:
    player_total, dealer_upcard, usable_ace = obs[:3]

    return (
        int(player_total),
        int(dealer_upcard),
        int(usable_ace)
    )

def counting_state_key(obs: np.ndarray):

    player_total, dealer_upcard, usable_ace, running_count, true_count, cards_remaining = obs

    return (
        int(player_total),
        int(dealer_upcard),
        int(usable_ace),

        int(np.clip(running_count, -20, 20)),
        int(np.clip(true_count, -10, 10)),
        int(np.clip(cards_remaining // 52, 0, 6)),
    )


class BaselineBlackjackAgent(QLearningBlackjackAgent):
    def __init__(
            self, 
            env: gym.Env,
            **kwargs):
        
        super().__init__(
            env=env,
            state_encoder=baseline_state_key,
            **kwargs,
        )
        
class CountingBlackjackAgent(QLearningBlackjackAgent):
    def __init__(
            self, 
            env: gym.Env,
            **kwargs):
        
        super().__init__(
            env=env,
            state_encoder=counting_state_key,
            **kwargs,
        )


In [ ]:
SEEDS = [1, 42, 123]
EPISODES_PER_SEED = 10_000_000
EVAL_SEED = 1234
EVAL_EPISODES = 10_000_000
CHECKPOINT_INTERVAL = 5_000_000
SAVE_FINAL_MODELS = True
MODEL_DIR = PROJECT_ROOT / "models"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

agent_config = {
    "learning_rate": 0.01,
    "initial_epsilon": 1.0,
    "final_epsilon": 0.1,
    "epsilon_decay": (1.0 - 0.1) / (EPISODES_PER_SEED * 0.6),
}

def make_env(seed: int, n_episodes: int):
    env = BlackjackEnv(
        num_decks=6,
        penetration=0.75,
        stand_on_soft_17=True,
    )

    env = gym.wrappers.RecordEpisodeStatistics(
        env,
        buffer_length=n_episodes,
    )

    env.reset(seed=seed)
    env.action_space.seed(seed)

    return env

agents = {}

for seed in SEEDS:
    agents[f"baseline-{seed}"] = BaselineBlackjackAgent(
        env=make_env(seed, EPISODES_PER_SEED),
        **agent_config,
    )
    agents[f"counting-{seed}"] = CountingBlackjackAgent(
        env=make_env(seed, EPISODES_PER_SEED),
        **agent_config,
    )

def split_agent_name(name: str) -> tuple[str, int]:
    agent_type, seed = name.rsplit("-", 1)
    return agent_type, int(seed)

def group_agents_by_type(agents):
    grouped = defaultdict(list)

    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        grouped[agent_type].append((seed, agent))

    return grouped


In [ ]:
# Checkpoint-Auswahl fuer Training und Evaluation
NO_ARTIFACT = None
IN_MEMORY_ARTIFACT = "__in_memory__"

def artifact_label(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

def artifact_options(agent_name: str, include_fresh: bool = False, include_memory: bool = False):
    options = []
    if include_fresh:
        options.append(("Neu trainieren (kein gespeichertes Modell laden)", NO_ARTIFACT))
    if include_memory:
        options.append(("Aktueller Agent im Speicher", IN_MEMORY_ARTIFACT))

    candidates = []
    candidates.extend(MODEL_DIR.glob(f"*{agent_name}*.pkl"))
    candidates.extend((CHECKPOINT_DIR / agent_name).glob("*.pkl"))
    candidates = sorted(
        {path.resolve(): path for path in candidates}.values(),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    options.extend((artifact_label(path), str(path)) for path in candidates)
    return options

TRAIN_ARTIFACT_SELECTORS = {
    name: widgets.Dropdown(
        options=artifact_options(name, include_fresh=True),
        value=NO_ARTIFACT,
        description=f"{name}:",
        layout=widgets.Layout(width="760px"),
    )
    for name in agents
}

display(widgets.VBox([widgets.HTML("<b>Training fortsetzen ab:</b>"), *TRAIN_ARTIFACT_SELECTORS.values()]))


# Training

In [ ]:
# Trainieren der Agenten
for name, agent in agents.items():
    agent_type, seed = split_agent_name(name)
    selected_artifact = TRAIN_ARTIFACT_SELECTORS[name].value
    start_episode = 0

    np.random.seed(seed)
    agent.env.reset(seed=seed)
    agent.env.action_space.seed(seed)

    if selected_artifact is not None:
        loaded_artifact = agent.load(selected_artifact)
        start_episode = int(loaded_artifact.get("episode") or 0)
        print(f"Lade {name} aus {selected_artifact}")
        print(f"-> Fortsetzung ab Episode {start_episode:,}")

    episodes_to_train = max(0, EPISODES_PER_SEED - start_episode)
    if episodes_to_train == 0:
        print(f"{name} ist bereits bei Episode {start_episode:,}; Training wird uebersprungen.\n")
        continue

    print(f"Starte Training fuer {name} ({episodes_to_train:,} Episoden)...")
    agent.train(
        n_episodes=episodes_to_train,
        base_seed=seed,
        start_episode=start_episode,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        checkpoint_dir=CHECKPOINT_DIR / name,
        checkpoint_label=f"{RUN_ID}_{name}_agent",
        checkpoint_metadata={
            "agent_name": name,
            "agent_type": agent_type,
            "run_id": RUN_ID,
            "seed": seed,
            "start_episode": start_episode,
            "target_episode": EPISODES_PER_SEED,
        },
    )
    print(f"Training fuer {name} erfolgreich beendet.\n")


# Saving

In [ ]:
timestamp = RUN_ID

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_FINAL_MODELS:
    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        print(f"Saving {name} agent...")
        
        filename = f"{timestamp}_{name}_agent.pkl"
        path = MODEL_DIR / filename
        
        agent.save(
            path,
            label=f"{name}_{timestamp}",
            artifact_type="final",
            episode=EPISODES_PER_SEED,
            n_episodes=EPISODES_PER_SEED,
            base_seed=seed,
            metadata={
                "agent_name": name,
                "agent_type": agent_type,
                "run_id": RUN_ID,
                "seed": seed,
                "target_episode": EPISODES_PER_SEED,
            },
            include_history=False,
        )
        print(f"-> Erfolgreich gespeichert unter: {path}")
else:
    print("Finales Speichern ist deaktiviert (SAVE_FINAL_MODELS=False).")


# Evaluation

## Greedy Policy Evaluation


In [ ]:
# Auswahl, welches Modell/Checkpoint evaluiert werden soll
EVAL_ARTIFACT_SELECTORS = {
    name: widgets.Dropdown(
        options=artifact_options(name, include_memory=True),
        value=IN_MEMORY_ARTIFACT,
        description=f"{name}:",
        layout=widgets.Layout(width="760px"),
    )
    for name in agents
}

display(widgets.VBox([widgets.HTML("<b>Evaluation-Modell auswaehlen:</b>"), *EVAL_ARTIFACT_SELECTORS.values()]))


In [ ]:
AGENT_CLASSES = {
    "baseline": BaselineBlackjackAgent,
    "counting": CountingBlackjackAgent,
}

def make_agent(agent_name: str, seed: int):
    agent_type, _ = split_agent_name(agent_name)
    return AGENT_CLASSES[agent_type](
        env=make_env(seed, EVAL_EPISODES),
        **agent_config,
    )

def selected_eval_agent(agent_name: str):
    selected_artifact = EVAL_ARTIFACT_SELECTORS[agent_name].value
    if selected_artifact == IN_MEMORY_ARTIFACT:
        return agents[agent_name], "in_memory"

    _, seed = split_agent_name(agent_name)
    eval_agent = make_agent(agent_name, seed=seed)
    artifact = eval_agent.load(selected_artifact)
    label = artifact.get("label") or artifact_label(Path(selected_artifact))
    return eval_agent, label

def greedy_action(agent, obs) -> int:
    state = agent.state_encoder(obs)
    values = agent.q_values.get(state)
    if values is None:
        return 0
    return int(np.argmax(values))

def evaluate_greedy_policy(agent, agent_name: str, n_episodes: int = EVAL_EPISODES, base_seed: int = EVAL_SEED) -> dict:
    env = make_env(base_seed, n_episodes)
    wins = losses = pushes = 0
    action_counts = {0: 0, 1: 0}

    for episode in tqdm(range(n_episodes), desc=f"eval {agent_name}", leave=False):
        obs, _ = env.reset(seed=base_seed + episode)
        terminated = False
        truncated = False
        episode_reward = 0.0

        while not (terminated or truncated):
            action = greedy_action(agent, obs)
            action_counts[action] = action_counts.get(action, 0) + 1
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward

        if episode_reward > 0:
            wins += 1
        elif episode_reward < 0:
            losses += 1
        else:
            pushes += 1

    total = wins + losses + pushes
    return {
        "episodes": total,
        "win_rate": wins / total if total else 0.0,
        "loss_rate": losses / total if total else 0.0,
        "push_rate": pushes / total if total else 0.0,
        "average_reward": (wins - losses) / total if total else 0.0,
        "action_distribution": {"stand": action_counts.get(0, 0), "hit": action_counts.get(1, 0)},
    }

greedy_eval_results = {}
for name in agents:
    eval_agent, source_label = selected_eval_agent(name)
    metrics = evaluate_greedy_policy(eval_agent, name)
    greedy_eval_results[name] = {"source": source_label, **metrics}

greedy_eval_df = pd.DataFrame(greedy_eval_results).T
display(greedy_eval_df)


## Training Dynamics

In [ ]:
def get_moving_stats(arr, window: int):
    arr = np.asarray(arr).flatten()
    if len(arr) == 0:
        return arr, arr, np.arange(0)

    min_periods = max(1, window // 10)
    series = pd.Series(arr)

    means = series.rolling(window=window, min_periods=min_periods).mean().to_numpy()
    stds = series.rolling(window=window, min_periods=min_periods).std().to_numpy()
    stds = np.nan_to_num(stds)

    return means, stds, np.arange(len(arr))

def plot_all(agents: dict, reward_window: int = 1000, td_window: int = 5000):

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    grouped = group_agents_by_type(agents)

    for agent_type, agent_list in grouped.items():

        style = AGENT_STYLES[agent_type]

        # ======================
        # REWARDS (correct)
        # ======================

        reward_curves = []

        for seed, agent in agent_list:

            rewards = np.asarray(agent.episode_rewards)
            means, _, _ = get_moving_stats(rewards, reward_window)

            reward_curves.append(means)

        # align lengths (IMPORTANT)
        min_len = min(len(c) for c in reward_curves)
        reward_curves = np.array([c[:min_len] for c in reward_curves])

        reward_mean = reward_curves.mean(axis=0)
        reward_std = reward_curves.std(axis=0)

        x_r = np.arange(len(reward_mean))

        axs[0].plot(
            x_r,
            reward_mean,
            label=style["label"],
            color=style["color"],
            linewidth=2.5,
        )

        axs[0].fill_between(
            x_r,
            reward_mean - reward_std,
            reward_mean + reward_std,
            color=style["color"],
            alpha=0.2,
        )

        # ======================
        # TD ERROR (correct)
        # ======================

        td_curves = []

        for seed, agent in agent_list:

            td_errors = np.asarray(agent.training_error)
            means, _, _ = get_moving_stats(td_errors, td_window)

            td_curves.append(means)

        # align lengths (IMPORTANT)
        min_len = min(len(c) for c in td_curves)
        td_curves = np.array([c[:min_len] for c in td_curves])

        td_mean = td_curves.mean(axis=0)
        td_std = td_curves.std(axis=0)

        x_t = np.arange(len(td_mean))

        axs[1].plot(
            x_t,
            td_mean,
            label=style["label"],
            color=style["color"],
            linewidth=2.5,
        )

        axs[1].fill_between(
            x_t,
            td_mean - td_std,
            td_mean + td_std,
            color=style["color"],
            alpha=0.2,
        )

    axs[0].set_title("Episode Rewards (Mittelwert über Seeds)")
    axs[0].set_xlabel("Episode")
    axs[0].set_ylabel("Reward")
    axs[0].legend()

    axs[1].set_title("TD Error (Mittelwert über Seeds)")
    axs[1].set_xlabel("Episode")
    axs[1].set_ylabel("TD Error")
    axs[1].legend()

    save_fig("training_results_combined")
    plt.show()

In [ ]:
plot_all(agents, reward_window=1000, td_window=5000)

## Performance Comparison

In [ ]:
print("Starte Trainings-Auswertung der Agenten...\n")

def get_reward_matrix(agent_list):
    return np.vstack([
        np.asarray(agent.episode_rewards)
        for seed, agent in agent_list
    ])


def flatten_rewards(agent_list):
    return get_reward_matrix(agent_list).ravel()


def calculate_metrics(rewards: np.ndarray) -> dict:
    rewards = np.asarray(rewards)

    wins = np.sum(rewards == 1)
    losses = np.sum(rewards == -1)
    pushes = np.sum(rewards == 0)
    total = len(rewards)

    return {
        "win_rate": wins / total if total > 0 else 0.0,
        "loss_rate": losses / total if total > 0 else 0.0,
        "push_rate": pushes / total if total > 0 else 0.0,
        "average_reward": np.mean(rewards) if total > 0 else 0.0,
        "std_reward": np.std(rewards) if total > 0 else 0.0,
    }


grouped = group_agents_by_type(agents)

per_seed_rows = []
for agent_type, agent_list in grouped.items():
    for seed, agent in agent_list:
        row = calculate_metrics(np.asarray(agent.episode_rewards))
        row.update({"agent_type": agent_type, "seed": seed})
        per_seed_rows.append(row)

per_seed_metrics_df = pd.DataFrame(per_seed_rows).sort_values(["agent_type", "seed"])
display(per_seed_metrics_df)

baseline_rewards = flatten_rewards(grouped["baseline"])
counting_rewards = flatten_rewards(grouped["counting"])

baseline_metrics = calculate_metrics(baseline_rewards)
counting_metrics = calculate_metrics(counting_rewards)

summary_metrics_df = pd.DataFrame({
    "baseline": baseline_metrics,
    "counting": counting_metrics,
}).T
display(summary_metrics_df)

for name, metrics in [("Baseline", baseline_metrics), ("Counting", counting_metrics)]:
    print(f"{name} metrics:")
    print(f"  Win Rate:       {metrics['win_rate']:.1%}")
    print(f"  Loss Rate:      {metrics['loss_rate']:.1%}")
    print(f"  Push Rate:      {metrics['push_rate']:.1%}")
    print(f"  Avg Reward:     {metrics['average_reward']:.3f}")
    print(f"  Std Reward:     {metrics['std_reward']:.3f}")
    print()


In [ ]:
if "greedy_eval_df" not in globals():
    raise RuntimeError("Bitte zuerst die Greedy-Evaluation ausfuehren.")

eval_metrics_df = greedy_eval_df.copy().reset_index().rename(columns={"index": "agent_name"})
name_parts = eval_metrics_df["agent_name"].apply(
    lambda name: pd.Series(split_agent_name(name), index=["agent_type", "seed"])
)
eval_metrics_df = pd.concat([eval_metrics_df, name_parts], axis=1)

metric_columns = ["win_rate", "loss_rate", "push_rate", "average_reward"]
for column in ["episodes", *metric_columns]:
    eval_metrics_df[column] = pd.to_numeric(eval_metrics_df[column])

display(eval_metrics_df[["agent_name", "agent_type", "seed", "source", "episodes", *metric_columns]])

eval_summary_df = eval_metrics_df.groupby("agent_type")[metric_columns].agg(["mean", "std"])
display(eval_summary_df)

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
axs = axs.ravel()

metric_plot_config = [
    ("win_rate", "Win Rate", (0, 0.6)),
    ("loss_rate", "Loss Rate", (0, 0.7)),
    ("push_rate", "Push Rate", (0, 0.2)),
    ("average_reward", "Average Reward", None),
]

agent_types = [agent_type for agent_type in ["baseline", "counting"] if agent_type in eval_metrics_df["agent_type"].unique()]
x = np.arange(len(agent_types))
colors = [AGENT_STYLES[agent_type]["color"] for agent_type in agent_types]
labels = [AGENT_STYLES[agent_type]["label"] for agent_type in agent_types]

for ax, (metric, title, ylim) in zip(axs, metric_plot_config):
    means = []
    stds = []

    for agent_type in agent_types:
        values = eval_metrics_df.loc[eval_metrics_df["agent_type"] == agent_type, metric]
        means.append(values.mean())
        stds.append(values.std(ddof=0) if len(values) > 1 else 0.0)

    ax.bar(x, means, yerr=stds, color=colors, width=0.45, capsize=6, alpha=0.85)

    for idx, agent_type in enumerate(agent_types):
        values = eval_metrics_df.loc[eval_metrics_df["agent_type"] == agent_type, metric].to_numpy()
        jitter = np.linspace(-0.08, 0.08, len(values)) if len(values) > 1 else np.array([0.0])
        ax.scatter(np.full(len(values), idx) + jitter, values, color="black", s=35, zorder=3, alpha=0.7)

    ax.set_title(f"{title} (Greedy Evaluation)")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(metric)
    if ylim is not None:
        ax.set_ylim(*ylim)

    for idx, value in enumerate(means):
        ax.text(idx, value, f"{value:.3f}", ha="center", va="bottom")

plt.tight_layout()
save_fig("final_evaluation_metrics")
plt.show()


# Policy & Q-Value Visualization
Hier analysieren wir interaktiv die gelernten Strategien der Agenten und vergleichen sie mit der Basic Strategy.

In [ ]:
from utils.visualizations import plot_policy_and_q_values

def interactive_plot_wrapper(agent_name, true_count):
    if agent_name in agents:
        plot_policy_and_q_values(
            agent=agents[agent_name], 
            agent_name=agent_name, 
            split_agent_name_func=split_agent_name, 
            save_fig_func=save_fig, 
            true_count=true_count
        )

agent_selector = widgets.Dropdown(options=list(agents.keys()), value=list(agents.keys())[0], description='Agent:')
true_count_slider = widgets.IntSlider(value=0, min=-10, max=10, step=1, description='True Count:', disabled=True)

def update_slider_visibility(*args):
    agent_type, _ = split_agent_name(agent_selector.value)
    true_count_slider.disabled = (agent_type == "baseline")

agent_selector.observe(update_slider_visibility, 'value')

widgets.interactive(interactive_plot_wrapper, agent_name=agent_selector, true_count=true_count_slider)